# epimux tutorialA complete bulk multi-omic differential analysis, from count matrices to areport — and, more importantly, the checks that decide whether the result canbe believed.We use synthetic data with a **known ground truth** so every claim can beverified against what we planted.

## 1. Build a datasetEverything is anchored on one reference element set. Each assay keeps its ownnative features and is projected onto that reference, so layers become directlycomparable without repeated interval arithmetic.

In [ ]:
import numpy as npimport pandas as pdimport epimux as eprng = np.random.default_rng(0)N, N_CHANGE = 4000, 300elements = pd.DataFrame({    "chrom": "chr1",    "start": np.arange(N) * 5_000,    "end":   np.arange(N) * 5_000 + 1_000,})def make_counts(effect, seed):    r = np.random.default_rng(seed)    base = r.lognormal(4.4, 0.9, N)    cols, design = {}, {"WT": [], "KO": []}    for g in ("WT", "KO"):        for i in (1, 2, 3):            mu = base.copy()            if g == "KO":                mu[:N_CHANGE] *= effect          # <-- the ground truth            cols[f"{g}_R{i}"] = r.poisson(mu)            design[g].append(f"{g}_R{i}")    return pd.DataFrame(cols, index=[f"e{i}" for i in range(N)]), designatac, design = make_counts(2.2, seed=1)h3k,  _      = make_counts(2.0, seed=2)ds = ep.Dataset(elements, genome="synthetic", name="tutorial")ds.add_counts("ATAC",    intervals=elements.set_index(atac.index), matrix=atac)ds.add_counts("H3K27ac", intervals=elements.set_index(h3k.index),  matrix=h3k)ds.set_design(design)ds.summary()

## 2. Differential analysis`log2FC` is **always** `log2(test / ref)`. The direction is carried by a`Contrast` object rather than inferred from factor levels — and it is re-verifiedagainst the raw counts before the result is returned.This matters more than it sounds. In R, `factor(c("WT","KO"))` sorts its levelsalphabetically to `("KO","WT")`, so a contrast written as "WT vs KO" silentlycomputes `log2(WT/KO)`. Every reported direction inverts and nothing downstreamcomplains.

In [ ]:
res = ds.differential(ref="WT", test="KO")res["ATAC"].head()

In [ ]:
# the planted elements (0-299) should be the significant, UP onessig = ds.significant("ATAC")print(f"significant: {len(sig)}")print(f"of which planted (index < 300): {(sig.index < 300).sum()}")print(f"median log2FC: {sig['log2FC'].median():+.2f}   (we planted 2.2x = {np.log2(2.2):+.2f})")

## 3. The audit`audit()` is the part that distinguishes a trustworthy result from a plausibleone. Each check corresponds to a way real analyses go wrong:| Check | Question ||---|---|| `check_direction` | Is the reported sign consistent with the raw data? || `positive_control` | Can this pipeline detect a difference we *know* exists? || `null_contrast` | Do replicates of the same group produce hits? || `efficiency_balance` | Could a technical bias manufacture this direction? || `replicate_reliability` | Are group means stable enough to correlate? || `outlier_replicates` | Is one sample driving everything? || `pvalue_diagnostic` | Does the model fit at all? |

In [ ]:
ds.audit(positive_control=("WT", "KO"), null_group="WT")ds.audit_report.to_frame()

A failing `positive_control` is the most important single result in this table:a pipeline that cannot find a difference you know is there **cannot support anull result**. Reporting "no change" from such a pipeline is a false negativedressed up as a finding.

## 4. Cross-layer couplingThe headline number in a multi-omic paper is usually a cross-layer correlation.It is also easy to get wrong: correlate two results built from *opposite*contrast orientations and a coordinated change reads as "decoupling".`couple()` refuses to run in that situation rather than returning a flippednumber.

In [ ]:
c = ds.coupling("ATAC", "H3K27ac")print(c)

In [ ]:
states = ds.classify()ep.concordance(states)

States are called from **significance in each layer**, never from the sign of asingle noisy difference. Sign-only state calls are the classic source ofirreproducible "discordant element" lists — shuffle the replicates and themembership changes.

## 5. Normalization when a global shift is possibleSize-factor methods assume most features do not change. When that assumptionbreaks — a genome-wide gain or loss — they absorb the effect and hand back aconfident null.

In [ ]:
ep.assess_global_shift(atac, ds.contrast("WT", "KO"))

In [ ]:
# with spike-ins the global magnitude survives normalizationspike = {"WT_R1": 1000, "WT_R2": 1100, "WT_R3": 950,         "KO_R1": 1020, "KO_R2": 980,  "KO_R3": 1050}sf = ep.spike_in_factors(spike)sf

## 6. PowerHow large an effect could this design detect, and how many replicates would theobserved effect have needed?

In [ ]:
ds.power("ATAC")

## 7. Export and report

In [ ]:
ds.report("tutorial_report.html")

In [ ]:
written = ds.export("tutorial_out")import jsonjson.load(open(written["manifest"]))["results"]

The manifest records the contrast, the sign convention, the audit outcome andthe per-assay summary — so the analysis can be checked months later withoutre-running it.## Where to go next* `ep.hic` — P(s) and its log-derivative, saddle plots, compartment strength,  APA, differential insulation. Hi-C needs no spike-in, so it is often the right  way to ask about the *functional consequence* of a binding change when the  ChIP itself is too shallow to quantify.* `ep.annotation` — TSS distance, genomic context, ROSE-style super-enhancers.* `ep.enrichment` — pathway, motif and interval-overlap enrichment, each with an  explicit background.* `ep.tracks` — browser-style locus panels and metaprofiles.